# Stage 5 T-side Tactical EDA

Vitality T-side Mirage tactical exploration from audited Gold outputs.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

BASE = Path('..')
EDA_DIR = BASE / 'data/gold/analysis/t_side_tactical_eda'
def load_table(name):
    return pd.read_parquet(EDA_DIR / f'{name}.parquet')

In [ ]:
overview = load_table('t_side_eda_overview')
site_distribution = load_table('t_side_site_distribution')
opponents = load_table('t_side_opponent_summary')
window_regions = load_table('t_side_window_region_summary')
window_utility = load_table('t_side_window_utility_summary')
no_plant = load_table('t_side_no_plant_summary')
deaths = load_table('t_side_death_summary')
bomb = load_table('t_side_bomb_carrier_summary')
signatures = load_table('t_side_progression_signature_summary')
audit = load_table('t_side_eda_audit')
overview

In [ ]:
ax = site_distribution.set_index('t_round_outcome')['round_count'].plot(kind='bar', color=['#3b82f6', '#ef4444', '#64748b', '#f59e0b'])
ax.set_title('Vitality T-side outcomes on Mirage')
ax.set_xlabel('Outcome')
ax.set_ylabel('Rounds')
plt.tight_layout()

## A vs B by temporal window

In [ ]:
ab_regions = window_regions[(window_regions['window_type'] == 'interval') & window_regions['t_round_outcome'].isin(['plant_A', 'plant_B'])]
ab_regions.sort_values(['window_start', 't_round_outcome', 'round_share_with_region'], ascending=[True, True, False]).groupby(['window_start', 't_round_outcome']).head(3)

In [ ]:
utility_by_window = window_utility[window_utility['window_type'] == 'interval'].groupby(['window_start', 'window_end', 't_round_outcome'])['total_utilities'].sum().unstack(fill_value=0)
utility_by_window.plot(marker='o', figsize=(10, 4))
plt.title('Utility events by interval window')
plt.xlabel('Window start (seconds)')
plt.ylabel('Utility events')
plt.tight_layout()

## No-plant failure context

In [ ]:
no_plant.sort_values('round_count', ascending=False).head(20)

In [ ]:
deaths[(deaths['t_round_outcome'] == 'no_plant') & (deaths['death_type'] == 'first_target_team_death')].sort_values('round_count', ascending=False).head(15)

## C4 and bomb carrier

In [ ]:
late_bomb = bomb[(bomb['is_late_round']) | bomb['window_start'].isin([95, 105])]
late_bomb.sort_values('round_count', ascending=False).head(20)

## Progression signatures

In [ ]:
signatures.sort_values('count', ascending=False).head(25)

In [ ]:
opponents.sort_values('total_t_side_rounds', ascending=False)

## Key findings to inspect manually

- Identify the first interval where A and B regional shares visibly diverge.
- Inspect recurrent no-plant failure contexts and first Vitality death regions.
- Compare late-round C4 regions in 95-105s and 105-115s.
- Review high-frequency progression signatures by opponent before defining model features.
- Use the feature catalog to exclude labels, post-plant fields, and final outcomes from future training.

In [ ]:
audit